# Fine-Tuning LLM (Phi-2) on MedReason Dataset

## Overview
This notebook demonstrates how to fine-tune a pre-trained Phi-2 model on the MedReason medical reasoning dataset using Hugging Face Transformers with Parameter Efficient Fine-Tuning (PEFT) via QLoRA (8-bit quantization).

## Table of Contents
1. [Setup Environment](#1-setup-environment)
2. [Load Dataset](#2-load-dataset)
3. [Data Formatting](#3-data-formatting)
4. [Model Loading](#4-model-loading)
5. [Base Model Evaluation](#5-base-model-evaluation)
6. [LoRA Configuration](#6-lora-configuration)
7. [Model Training](#7-model-training)
8. [Fine-Tuned Model Evaluation](#8-fine-tuned-model-evaluation)
9. [Model Saving](#9-model-saving)

## 1. Setup Environment
### Prerequisites
- Python virtual environment
- Required Python packages


## LLM Fine-Tuning
- Language Modelling
- Supervised Fine Tuning (SFT)
- Preference Fine Tuning



### 1. Creating and Activate Virtual Environment & Install Required Packages 
- Create virtual environment using `virtualenv`
- Activate virtual environment  
- Install `uv` package manager for faster dependency resolution
- Install required packages:
    - accelerate: For distributed training
    - bitsandbytes: For 8-bit quantization
    - trl: For fine-tuning language models
    - peft: For parameter efficient fine-tuning 
    - transformers: For working with transformer models
    - datasets: For loading and processing datasets
    - huggingface_hub: For model/dataset access


In [0]:
%pip install -q -U transformers datasets accelerate peft trl bitsandbytes huggingface_hub[hf_xet] mlflow

In [0]:
dbutils.library.restartPython()

In [0]:
import os
import torch

if not torch.cuda.is_available():
    raise RuntimeError("GPU compute is required for Phi-2 QLoRA fine-tuning. Attach GPU-enabled compute before running this notebook.")

os.environ["TOKENIZERS_PARALLELISM"] = "false"
torch.backends.cuda.matmul.allow_tf32 = True

In [0]:
from pathlib import Path

base_model_id = "microsoft/phi-2"
max_length = 384
sample_size = 200
output_dir = str(Path.cwd() / "artifacts" / "phi2_medreason_qlora")

### 2. Load Dataset

Load the MedReason dataset, a large-scale, high-quality medical reasoning dataset designed to enable faithful and explainable medical problem-solving in large language models (LLMs).

**Note:** For faster training and due to GPU constraints, we only used the first 200 records from the dataset.

In [0]:
import pandas as pd
from datasets import Dataset

source_path = "hf://datasets/UCSC-VLAA/MedReason/ours_quality_33000.jsonl"
df = pd.read_json(source_path, lines=True, nrows=sample_size)
display(df.head(5))

In [0]:
print(f"Loaded {len(df):,} rows and {len(df.columns)} columns")

In [0]:
display(pd.DataFrame({"column": df.columns, "dtype": df.dtypes.astype(str)}))

In [0]:
dataset = Dataset.from_pandas(df, preserve_index=False).train_test_split(test_size=0.1, seed=0)
print({split: dataset[split].num_rows for split in dataset})

In [0]:
display(dataset["train"].to_pandas().head(5))

In [0]:
display(pd.DataFrame([dataset["test"][0]]))

## 3. Load Base Model and Prepare Formatting


Let's load the Phi-2 model and prepare the data formatting pipeline.

In [0]:
def build_prompt(question, options, answer, reasoning):
    return (
        "Given the question and option generate an answer and reasoning.\n"
        f"### question: {question}\n"
        f"### option: {options}\n\n"
        f"### answer with reasoning: {answer} and {reasoning}"
    )


def formatting_func(example):
    return build_prompt(
        example["question"],
        example["options"],
        example["answer"],
        example["reasoning"],
    )

In [0]:
print(formatting_func(dataset["train"][0]))

### 4. Load Base Model and Tokenize

In [0]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

In [0]:
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
    add_eos_token=True,
    add_bos_token=True,
)

tokenizer.pad_token = tokenizer.eos_token
tokenizer.model_max_length = max_length

In [0]:
def tokenize(batch):
    texts = [
        build_prompt(question, options, answer, reasoning)
        for question, options, answer, reasoning in zip(
            batch["question"],
            batch["options"],
            batch["answer"],
            batch["reasoning"],
        )
    ]
    return tokenizer(
        texts,
        truncation=True,
        max_length=max_length,
        padding="max_length",
    )

In [0]:
tokenized_preview = tokenize(dataset["train"][:1])
print({key: len(value[0]) for key, value in tokenized_preview.items()})
print(tokenizer.decode(tokenized_preview["input_ids"][0][:120], skip_special_tokens=False))

In [0]:
dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing MedReason samples",
)

dataset

## 5. Testing Base Model Performance without Fine-tuning

In [0]:
eval_prompt = """
Given the question and option generate a answer and reasoning.
### question : Urogenital Diaphragm is made up of the following, except:
### option : Answer Choices:
                A. Deep transverse Perineus
                B. Perinial membrane
                C. Colle's fascia
                D. Sphincter Urethrae

### answer with reasoning:
"""

In [0]:
def generate_response(model_to_use, tokenizer_to_use, prompt, max_new_tokens=256):
    model_input = tokenizer_to_use(prompt, return_tensors="pt").to("cuda")
    with torch.inference_mode():
        output = model_to_use.generate(
            **model_input,
            max_new_tokens=max_new_tokens,
            repetition_penalty=1.15,
            pad_token_id=tokenizer_to_use.eos_token_id,
        )
    return tokenizer_to_use.decode(output[0], skip_special_tokens=True)

In [0]:
model.eval()
print(generate_response(model, tokenizer, eval_prompt))

## 6. LoRA Configuration
- Configure the 8-bit QLoRA settings for parameter-efficient fine-tuning.

In [0]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

target_modules = ["Wqkv", "fc1", "fc2"]

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=target_modules,
    bias="none",
    lora_dropout=0.05,
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, config)
model.config.use_cache = False

In [0]:
def print_trainable_parameters(model):
    """
    Prints the number of trainable parameters in the model.
    """
    trainable_params = 0
    all_param = 0
    for _, param in model.named_parameters():
        all_param += param.numel()
        if param.requires_grad:
            trainable_params += param.numel()
    print(
        f"trainable params: {trainable_params} || all params: {all_param} || trainable%: {100 * trainable_params / all_param}"
    )

In [0]:
print_trainable_parameters(model)

## 7. Training the Model

In [0]:
import mlflow

mlflow.autolog(log_models=False, silent=True)

In [0]:
from datetime import datetime
from pathlib import Path
from transformers import DataCollatorForLanguageModeling, Trainer, TrainingArguments

Path(output_dir).mkdir(parents=True, exist_ok=True)
run_name = f"phi2-medreason-{datetime.now().strftime('%Y%m%d_%H%M%S')}"

args = TrainingArguments(
    output_dir=output_dir,
    run_name=run_name,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    optim="paged_adamw_8bit",
    fp16=True,
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to=["mlflow"],
    remove_unused_columns=False,
    group_by_length=True,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False),
)

trainer.train()

## 8. Testing the Fine-Tuned Model | Loading the PEFT Model

The PEFT library saves only the QLoRA adapters by default. To test the fine-tuned model, we first need to load the base model from the Hugging Face Hub and then merge it with the PEFT adapters.


In [0]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True,
)

eval_tokenizer = AutoTokenizer.from_pretrained(
    base_model_id,
    trust_remote_code=True,
    use_fast=False,
    padding_side="left",
    add_bos_token=True,
)
eval_tokenizer.pad_token = eval_tokenizer.eos_token

In [0]:
from pathlib import Path
from peft import PeftModel

checkpoints = sorted(
    Path(output_dir).glob("checkpoint-*"),
    key=lambda path: int(path.name.split("-")[-1]),
)
if not checkpoints:
    raise FileNotFoundError(f"No checkpoints found in {output_dir}")

latest_checkpoint = checkpoints[-1]
ft_model = PeftModel.from_pretrained(base_model, str(latest_checkpoint))

In [0]:
ft_model.eval()
print(generate_response(ft_model, eval_tokenizer, eval_prompt, max_new_tokens=400))


## 9. Saving the Fine-Tuned Model

To save the fine-tuned model for further use, we compress the QLoRA adapter checkpoint into a zip file.


In [0]:
import shutil

archive_path = shutil.make_archive("phi2_qlora_adapter", "zip", root_dir=str(latest_checkpoint))
print(f"Saved adapter archive to {archive_path}")